In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import pickle
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.lines as mlines
from pathlib import Path

from openquake.hazardlib.imt import PGA, SA, RSD595, AvgSA, IMT

from pickagm.distributions import ensemble_ecdfs

from phd_project.config.config import load_config 
from phd_project.scripts.WP1_ground_motion_set.gm_selection import (
    ESHM20SiteRupCtxBuilder,
    create_gmm_map,
    create_corr_model_map,
    calculate_site_gcim_distributions_for_all_sites,
    get_record_ensembles_for_site_and_poe,
    get_record_ensembles_for_sites,
    get_record_ensembles_for_single_site,
    get_gcim_distributions_for_single_site_and_poe,
    get_record_ensembles_for_single_distribution,
    get_imtl_from_disaggstats,
    get_imtl_from_disaggstats,
    get_gcim_distributions_for_single_site
)
import phd_project.scripts.WP1_ground_motion_set.manage_flatfiles as mf
from phd_project.plotting.plotting import custom_log_formatter

cfg = load_config()

In [3]:
# Load the disaggregation data
fp = cfg["proc_data"]["site_hazard"] / "AvgSA_06_disagg_data_60sites.pickle"
with open(fp, "rb") as f:
    disagg_data = pickle.load(f)

fp = cfg["proc_data"]["site_hazard"] / "AvgSA_06_disagg_stats_60sites.pickle"
with open(fp, "rb") as f:
    disagg_stats = pickle.load(f)

# load the site file
sites = pd.read_csv(cfg["hazard_models"]["eshm20_AvgSA_site_model_all"])

# load the flatfiles
flatfile_folder = cfg["proc_data"]["corr_model"] / "reverse" / "flatfiles"
flatfiles = {}
for f in [f for f in os.listdir(flatfile_folder) if f.endswith(".csv")]:
    tag = f.split("_")[0]
    flatfiles[tag] = pd.read_csv(flatfile_folder / f, delimiter=";", index_col=0, low_memory=False)

flatfiles["volcanic"] = pd.read_csv(cfg["raw_data"]["gm_flatfiles"] / "volcanic_lanzanoluzi_flatfile.csv", 
                                    delimiter=";", index_col=0)

# load the preprocessed gm database
gm_database = pd.read_csv(cfg["proc_data"]["gm_database"], sep=",", low_memory=False, header=[0, 1])

In [4]:
# create the average depth map for each TRT #TODO:: if this needs to be more specific
average_depths = {
    "Craton": flatfiles["asc"]["ev_depth_km"].mean(),
    "Non-Subduction Deep": flatfiles["vran"]["ev_depth_km"].mean(),
    "Shallow Default": flatfiles["asc"]["ev_depth_km"].mean(),
    "Subduction Inslab": flatfiles["sinter"]["ev_depth_km"].mean(),
    "Subduction Interface": flatfiles["sinter"]["ev_depth_km"].mean(),
    "Volcanic": flatfiles["volcanic"]["ev_depth_km"].mean(),
}

# the map of what sim trts are allowed to match with what record trts
OK_TRT_MATCHES = {
    "Craton": ["Shallow Default"],
    "Non-Subduction Deep": ["Non-Subduction Deep", "Subduction Inslab", "Subduction"],
    "Shallow Default": ["Shallow Default"],
    "Subduction Inslab": ["Non-Subduction Deep", "Subduction Inslab", "Subduction"],
    "Subduction Interface": ["Subduction Interface"],
    "Volcanic": ["Shallow Default"],
}

occurence = True    # the record selection should be performed based on occurence

# AvgSA([0, 6])

In [18]:
# set some parameters for the selection
t_lower = 0.025     # lower SA period considered in selection
t_upper = 6         # upper SA period considered in selection
n_periods = 20      # number of periods to consider in selection

conditioning_imt: IMT = AvgSA([0,6])                      
nonSA_imts: list[IMT] = [AvgSA([0,6]), RSD595(), PGA()] 
sa_periods = np.round(np.geomspace(t_lower, t_upper, num=n_periods), 3)
SA_imts: list[IMT] = [SA(period) for period in sa_periods]
selection_imts: list[IMT] = nonSA_imts[1:] + SA_imts    # not AvgSA but the others (PGA and RSD595)
nonSA_imt_strs: list[str] = [im.string for im in nonSA_imts] # strings match the correlation matrix

# weights of the IMs
weight_rsd595 = 0.3
n_other_ims = len([imt for imt in selection_imts if imt.name == "SA" or imt.name == "PGA"])
imt_weights = np.array([(1-weight_rsd595) / n_other_ims if imt.name != "RSD595" 
                        else weight_rsd595 for imt in selection_imts])
imt_weights /= imt_weights.sum()

# some other things
disagg_type = "TRT_Mag_Dist_Eps"
percentiles = [0.05, 0.16, 0.5, 0.84, 0.95]     # percentiles of the gcim distribution to return  
assumed_rake = 0                                # assumed rake for RSD595 calculation

In [19]:
# filter the gm_database so that only the selection and conditioning ims are present
gm_db = gm_database.copy()
updated_ims = mf.filter_gm_database_on_imts(
    gm_db["ims"], selection_imts + [conditioning_imt])
updated_ims.columns = pd.MultiIndex.from_product([['ims'], updated_ims.columns])
gm_db = pd.concat([gm_db.drop('ims', axis=1, level=0), updated_ims], axis=1)

# Create the GMM Map for AvgSA by reading the logic tree
AvgSA_06_lt_fp = cfg["hazard_models"]["eshm20_AvgSA"] / "gmpe_logic_tree_AvgSA_0to6_median_branch.xml"
gmm_map = create_gmm_map(AvgSA_06_lt_fp)

# get the correlation model map
corr_map = create_corr_model_map(nonSA_imt_strs, sa_periods)

In [20]:
# set up the selection context:
selection_ctx = {
    "n_ensembles": 20,
    "n_samples": 30,
    "conditioning_imt": conditioning_imt ,
    "disagg_imt": conditioning_imt.name , # this only works for AvgSA. otherwise used .string 
    "selection_imts": selection_imts ,
    "imt_weights": imt_weights ,
    "sites": sites ,
    "ctx_builder": ESHM20SiteRupCtxBuilder ,
    "ctx_builder_params": ["average_depths", "assumed_rake"] ,
    "average_depths": average_depths ,
    "assumed_rake": assumed_rake ,
    "gmm_map": gmm_map ,
    "corr_map": corr_map ,
    "m_bound_model": "tarbali_and_bradley_2016" ,
    "d_bound_model": "tarbali_and_bradley_2016" ,
    "vs30_bound_model": "tarbali_and_bradley_2016" ,
    "sf_bounds": None,
    "usable_T": t_upper ,
    "max_n_recs": 3 ,
    "p_value": 0.05 ,
    "ok_trt_matches": OK_TRT_MATCHES ,
    "occurence": True ,
}
rng_seed = 2

## GCIM Distributions

In [21]:
LOAD_GCIM_IF_EXISTS = True
gcim_dist_fp = cfg["proc_data"]["gcim_dists"] / f"gcim_dist_AvgSA_06_rake{int(assumed_rake)}.pickle"

if LOAD_GCIM_IF_EXISTS: # load the gcim distributions instead of 
    no_file = False
    if gcim_dist_fp.is_file():
        with open(gcim_dist_fp, "rb") as file:
            gcim_distributions = pickle.load(file)
        print("Existing GCIM distribution data loaded...")
    else:
        no_file = True
        print("No existing GCIM distribution data found...")

if (not LOAD_GCIM_IF_EXISTS) or no_file:
    # calculate the gcims and save them
    gcim_distributions = calculate_site_gcim_distributions_for_all_sites(
        disagg_data, disagg_stats, conditioning_imt, 
        selection_imts, sites, gmm_map, corr_map, 
        average_depths, assumed_rake, occurence, percentiles)

    with open(gcim_dist_fp, "wb") as file:
        pickle.dump(gcim_distributions, file)

No existing GCIM distribution data found...
Calculating GCIM Distributions...
  Region 0 - high seismicity
    site id: 30
    site id: 31
    site id: 32
    site id: 33
    site id: 34
  Region 1 - high seismicity
    site id: 35
    site id: 36
    site id: 37
    site id: 38
    site id: 39
  Region 2 - high seismicity
    site id: 40
    site id: 41
    site id: 42
    site id: 43
    site id: 44
  Region 3 - high seismicity
    site id: 45
    site id: 46
    site id: 47
    site id: 48
    site id: 49
  Region 4 - high seismicity
    site id: 50
    site id: 51
    site id: 52
    site id: 53
    site id: 54
  Region 5 - high seismicity
    site id: 55
    site id: 56
    site id: 57
    site id: 58
    site id: 59
  Region 0 - lowmod seismicity
    site id: 0
    site id: 1
    site id: 2
    site id: 3
    site id: 4
  Region 1 - lowmod seismicity
    site id: 5
    site id: 6
    site id: 7
    site id: 8
    site id: 9
  Region 2 - lowmod seismicity
    site id: 10
    site 

## Selection

In [22]:
LOAD_RS_IF_EXISTS = True
selection_output_fp = cfg["proc_data"]["gm_selection"] / f"AvgSA_06_raw_selection_results_rake{int(assumed_rake)}.pickle"
only_sites = [30, 31]
only_poes = [0.000404, 0.0001]


if LOAD_RS_IF_EXISTS: # load the gcim distributions instead of 
    no_file = False
    if selection_output_fp.is_file():
        with open(selection_output_fp, "rb") as file:
            record_selection_results = pickle.load(file)
        print("Existing record selection data loaded...")
    else:
        no_file = True
        print("No existing record selection data found...")
        
if (not LOAD_RS_IF_EXISTS) or no_file:
    # do the record selection for all sites and save the results
    record_selection_results = get_record_ensembles_for_sites(
        disagg_data, disagg_stats, gcim_distributions, 
        gm_db, selection_ctx, rng_seed, only_sites, only_poes    
        )

    # Save the results
    with open(selection_output_fp, "wb") as file:
        pickle.dump(record_selection_results, file)

No existing record selection data found...
Selecting Record Ensembles...
  Region 0 - high seismicity
    site id: 30
        poe: 0.000404
        poe: 0.0001
    site id: 31
        poe: 0.000404
        poe: 0.0001
  Region 1 - high seismicity
  Region 2 - high seismicity
  Region 3 - high seismicity
  Region 4 - high seismicity
  Region 5 - high seismicity
  Region 0 - lowmod seismicity
  Region 1 - lowmod seismicity
  Region 2 - lowmod seismicity
  Region 3 - lowmod seismicity
  Region 4 - lowmod seismicity
  Region 5 - lowmod seismicity


In [23]:
# Check if all the sites and poes found a set of records successfully
no_ensemble_found = []
for (s, r) in record_selection_results.keys():
    for site_id in record_selection_results[(s,r)].keys():
        for poe, rsr in record_selection_results[(s,r)][site_id].items():
            if not rsr["ensemble_found"]:
                no_ensemble_found.append(((s,r), site_id, poe))

if {len(no_ensemble_found)}:
    print(f"OK! - Records found for all combinations of site and poe")
else:
    print(f"Sub-Optimal! - No records found for {len(no_ensemble_found)} combinations of site and poe")
    for ii in no_ensemble_found:
        print(ii)

OK! - Records found for all combinations of site and poe


In [ ]:
# im = "RSD595"
# poe1 = 0.000404
# poe2 = 0.0001
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3), sharex=True, sharey=True)

# # Figure 1 -> RSD595 KS-Test poe1
# records1 = record_selection_results_AvgSA06[poe1]
# if records1["ensemble_found"]:
#     ensemble1 = records1["best_ensemble"]
#     ecdfs1 = ensemble_ecdfs(ensemble1["recs"]["ims_scaled"])
# else:
#     rscores = np.array([r["R-score"] for r in records1["all_ensembles"]])
#     idx = np.where(rscores == min(rscores))[0][0]
#     ensemble1 = records1["all_ensembles"][idx]
#     print(f"KS-Test failed for poe {poe1}. Showing ensemble with R = {ensemble1["R-score"]:.4f} and failed IMs {ensemble1["ks_failed_ims"]}")
#     ecdfs1 = ensemble_ecdfs(ensemble1["recs"]["ims_scaled"])

# # target cdf and ks bounds
# ks_bounds = records1["ks_bounds"][im]
# xs = ks_bounds[:, 0]
# lower = ks_bounds[:, 1]
# cdf = ks_bounds[:, 2]
# upper = ks_bounds[:, 3]

# # ecdf of records
# im_ecdf = ecdfs1[im]

# ax1.plot(xs, lower, color="0.8", ls="--")
# ax1.plot(xs, cdf, color="0.8", ls="-")
# ax1.plot(xs, upper, color="0.8", ls="--")
# ax1.plot(im_ecdf[:,0], im_ecdf[:,1], color="b")
# # ax1.grid(True, which="both", ls="-.", color="0.8")
# # ax1.minorticks_on()

# # Figure 2 -> RSD595 KS-Test poe2
# records2 = record_selection_results_AvgSA06[poe2]
# if records2["ensemble_found"]:
#     ensemble2 = records2["best_ensemble"]
#     ecdfs2 = ensemble_ecdfs(ensemble2["recs"]["ims_scaled"])
# else:
#     rscores = np.array([r["R-score"] for r in records2["all_ensembles"]])
#     idx = np.where(rscores == min(rscores))[0][0]
#     ensemble2 = records2["all_ensembles"][idx]
#     print(f"KS-Test failed for poe {poe2}. Showing ensemble with R = {ensemble2["R-score"]:.4f} and failed IMs {ensemble2["ks_failed_ims"]}")
#     ecdfs2 = ensemble_ecdfs(ensemble2["recs"]["ims_scaled"])

# # target cdf and ks bounds
# ks_bounds = records["ks_bounds"][im]
# xs = ks_bounds[:, 0]
# lower = ks_bounds[:, 1]
# cdf = ks_bounds[:, 2]
# upper = ks_bounds[:, 3]

# # ecdf of records
# im_ecdf = ecdfs2[im]

# ax2.plot(xs, lower, color="0.8", ls="--")
# ax2.plot(xs, cdf, color="0.8", ls="-")
# ax2.plot(xs, upper, color="0.8", ls="--")
# ax2.plot(im_ecdf[:,0], im_ecdf[:,1], color="b")
# ax2.set_xlim(0, 80)
# ax2.set_ylim(0, 1.0)
# # ax2.set_xlabel(r"$D_{s,5\%-95\%}$ [s]")
# ax2.set_ylabel(r"P[IM <= im]")
# # ax2.grid(True, which="both", ls="-.", color="0.8")
# # ax2.minorticks_on()
# fig.tight_layout()


# Plotting